In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from instruments.gilson.gilson_ethernet import GilsonEthernet
from instruments.gilson.tray import Tray
from instruments.gilson.rack import Rack_209, Rack_3dp
from instruments.gilson.probe import ProbeState    # NEW
from core.logging import flow_logger as logger
from instruments.vici.dim import DIM
from instruments.vici.fsm import FSM
from instruments.syr_pumps.HB_syr_pump import HBElite
from instruments.knauer.knauer_pump_azura import KnauerPumpAzura
from ecosystems.reaction_segment_generation import RSG
from ecosystems.segmentation import Segmentation


# Experiment framework
from core.experiment_context import ExperimentManager
from core.experiment_builder import ExperimentBuilder
from core.experiment_validator import ExperimentValidator
from core.experiment_compiler import ExperimentCompiler
from core.experiment_intent import ExperimentIntent
from core.inventory import Inventory

In [2]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4


In [3]:
# --- Instantiate the tray ---
tray = Tray()

# --- Instantiate modules for the tray ---
rack1 = Rack_209()
rack2 = Rack_3dp()

# --- Assign modules to tray slots ---
tray.assign_slot(1, rack1, alias = "rack1")    # Assigns rack1 to slot 1
tray.assign_slot(2, rack2, alias = "rack2")    # Assigns rack2 to slot 2
tray.assign_slot(3, dim, alias = "dim")      # Assigns dim to slot 3

In [4]:
# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Instantiate the probe (state machine only) ---
probe = ProbeState()

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("VB-1E-9")

In [5]:
# Instantiate the RSG ecosystem with the connected Gilson, pump, DIM and probe
rsg = RSG(gilson=g, pump=syr_pump, dim=dim, probe=probe)

# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, fsm=fsm, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [ ]:
inventory = Inventory()

# Clear + reassign (optional but safer for now)
inventory.assign(
    module="rack2",
    vial=1,
    name="MeCN",
    concentration_M=None,
    solvent="MeCN",
    current_volume_uL=40000,
    min_safe_volume_uL=20000
)

inventory.assign(
    module="rack1",
    vial=1,
    name="BnOH",
    concentration_M=None,
    solvent="MeCN",
    current_volume_uL=1500,
    min_safe_volume_uL=500
)

In [ ]:
intent = ExperimentIntent(
    experiment_id="VB-1E-9A",
    description="Series of 20 slugs of BnOH (concurrent sampling valve plumbing + 1mL/min)"
)

intent.add_block(
    name="20 identical BnOH slugs",
    components=["BnOH"],
    ratios=[[100]],  
    total_volume_uL=100.0
)

intent.summary()

In [ ]:
compiler = ExperimentCompiler(inventory)

compiled_df = compiler.compile(intent)

compiled_df

In [ ]:
builder = ExperimentBuilder(inventory=inventory)

result = builder.build_and_create(
    experiment_id="VB-1E-9A",
    rows=compiled_df,
    description=intent.description,
    global_conditions={
        "flowrate_ul_min": 1000,
        "gas_prime_s": 20,
        "withdraw_rate_ml_min": 0.25,
        "dispense_rate_ml_min": 0.5,
    },
    overwrite=True
)

print(result["plan_path"])

pd.DataFrame(result["summary"])

In [ ]:
manager = ExperimentManager()

manager.load_experiment("VB-1E-9A")

manager.status()

In [ ]:
trace = manager.preview_execution(
    seg,
    wash_policy="needle_only",
)

for step in trace:
    print(step)

In [ ]:
manager.precondition_reactor(seg, carrier_flowrate_ul_min=1000, stabilisation_time_s=120)

manager.prepare_rsg(
    seg,
    air_gap=20.0,
    rate_wdr=0.10
)

In [ ]:
manager.arm_experiment()

In [ ]:
manager.execute_experiment(
    seg,
    wash_policy="needle_only",
)